# 单变量 vs 多变量对比实验

本 notebook 对应 `scripts/run_univariate_multivariate.py`。它把同一组数据拆成两种训练口径：

- **单变量**：只输入目标列，并只预测目标列。
- **多变量**：输入全部变量，并预测全部变量，报告时重点看目标列指标。

默认配置覆盖 ETTh1/ETTm1、h96/h336、LSTM/Transformer/Autoformer/PatchTST。为了便于课堂复现，配置文件默认使用小样本；如需全量实验，将 `configs/univariate_multivariate_comparison.json` 中的 `sample_limit` 改为 `0`。

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

CONFIG_PATH = ROOT / 'configs' / 'univariate_multivariate_comparison.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
config

## 1. 查看原始多变量数据形状

预处理后的 `X` 形状为 `(样本数, 回看窗口, 变量数)`，`Y` 形状为 `(样本数, 预测步长, 变量数)`。ETT 数据集有 7 个变量。

In [ ]:
from models import TimeSeriesDataset

dataset_name = 'ETTh1'
horizon = 96
data_dir = ROOT / config['data_dir']

multi_dataset = TimeSeriesDataset(data_dir, dataset_name, horizon, 'train')
x, y = multi_dataset[0]

print('target_idx:', multi_dataset.target_idx)
print('multivariate X:', tuple(x.shape))
print('multivariate Y:', tuple(y.shape))

## 2. 单变量口径如何得到

单变量实验不重新预处理文件，而是在训练时只切出目标列。这样可以保证单变量和多变量使用完全相同的时间切分与标准化参数。

In [ ]:
target_idx = multi_dataset.target_idx
x_uni = x[:, target_idx:target_idx + 1]
y_uni = y[:, target_idx:target_idx + 1]

print('univariate X:', tuple(x_uni.shape))
print('univariate Y:', tuple(y_uni.shape))

## 3. 运行专用训练脚本

下面命令会根据配置依次运行 `univariate` 和 `multivariate` 两种模式，并在 `results/v1_csv/feature_mode/` 与 `results/v1_md/feature_mode/` 中生成对比表。

In [ ]:
!python {ROOT / 'scripts' / 'run_univariate_multivariate.py'} --config {CONFIG_PATH}

## 4. 读取对比表

`*_comparison.csv` 是完整明细；`*_comparison_delta.csv` 直接比较目标列指标，其中 `delta = 单变量 - 多变量`。因此 `delta_MSE_target < 0` 表示单变量目标列 MSE 更低。

In [ ]:
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError('请先执行 pip install -r requirements.txt 安装 pandas 后再运行展示单元。') from exc

run_tag = config['run_tag']
detail_path = ROOT / 'results' / 'v1_csv' / 'feature_mode' / f'{run_tag}_comparison.csv'
delta_path = ROOT / 'results' / 'v1_csv' / 'feature_mode' / f'{run_tag}_comparison_delta.csv'

detail = pd.read_csv(detail_path)
delta = pd.read_csv(delta_path)

display(detail.head())
display(delta)

## 5. 更直观的透视表

下面把目标列 MSE 按变量模式展开，便于直接观察哪些组合更适合单变量输入，哪些组合更依赖多变量信息。

In [ ]:
pivot = detail.pivot_table(
    index=['dataset', 'horizon', 'model'],
    columns='feature_mode',
    values='MSE_target',
)
pivot['univariate_minus_multivariate'] = pivot['univariate'] - pivot['multivariate']
display(pivot.reset_index())